# schedule dashboard page refresh v1

目标：验证 `src/schedule` 可以把模型矩阵刷新、dashboard export、view model 和 HTML dashboard 页面串成一条可定时执行的链路。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'SPEC.md').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot locate repo root from notebook cwd')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import build_dashboard_view_model
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = Path('research/schedule_dashboard_page_refresh_v1')
MATRIX_CARD = CARD / 'matrix_card'
DASHBOARD_EXPORT = CARD / 'dashboard_export.json'
DASHBOARD_HTML = CARD / 'dashboard.html'
ATTRIBUTION_CARD = Path('research/model_attribution_analysis_v1')
NOTEBOOK_CARD_MANIFEST = Path('research/notebook_card_manifest_v1/notebook_card_manifest.json')
PRODUCTION_HEALTH = Path('research/production_registry_health_v1/production_health_export.json')
LOCK_DIR = CARD / 'locks'
CARD.mkdir(parents=True, exist_ok=True)

{'card': str(CARD), 'dashboard_html': str(DASHBOARD_HTML), 'notebook_card_manifest': str(NOTEBOOK_CARD_MANIFEST), 'production_health': str(PRODUCTION_HEALTH)}

{'card': 'research/schedule_dashboard_page_refresh_v1',
 'dashboard_html': 'research/schedule_dashboard_page_refresh_v1/dashboard.html',
 'notebook_card_manifest': 'research/notebook_card_manifest_v1/notebook_card_manifest.json',
 'production_health': 'research/production_registry_health_v1/production_health_export.json'}

## 执行 schedule refresh

In [2]:
config = ModelDashboardPipelineConfig(
    name='schedule_dashboard_page_refresh_v1',
    card_dir=MATRIX_CARD,
    dashboard_output_path=DASHBOARD_EXPORT,
    dashboard_page_output_path=DASHBOARD_HTML,
    dashboard_page_title='SOXL 模型回测 Dashboard',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
    top_n_factors=3,
    dashboard_top_n=5,
    attribution_card_dir=ATTRIBUTION_CARD,
    notebook_card_manifest_path=NOTEBOOK_CARD_MANIFEST,
    production_health_path=PRODUCTION_HEALTH,
    retries=1,
)
events = []
result = run_model_dashboard_refresh_once(config, lock_dir=LOCK_DIR, notify=events.append)
{'status': result.status, 'payload': result.payload, 'events': events}

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'success',
 'payload': {'matrix_card': 'research/schedule_dashboard_page_refresh_v1/matrix_card',
  'dashboard_export': 'research/schedule_dashboard_page_refresh_v1/dashboard_export.json',
  'dashboard_schema_version': 'dashboard.model_matrix.v1',
  'dashboard_artifact_id': 'matrix_card',
  'dashboard_manifest_kind': 'analysis_manifest',
  'dashboard_manifest_path': 'research/schedule_dashboard_page_refresh_v1/matrix_card/analysis_manifest.json',
  'dashboard_manifest_hash': '6422a0b45fd294e4ccd84ab9f7f59161ea26174ebbed1c58e51dda48703e1941',
  'dashboard_manifest_producer': 'src.analysis.exports',
  'dashboard_page': 'research/schedule_dashboard_page_refresh_v1/dashboard.html',
  'dashboard_page_schema_version': 'dashboard.static_page.v1',
  'dashboard_view_model_schema_version': 'dashboard.view_model.v1',
  'dashboard_page_model_card_count': 5,
  'top_model': 'ridge_mse__ic_weighted_composite',
  'attribution_card': 'research/model_attribution_analysis_v1',
  'attribution_s

## 验证页面链路

In [3]:
dashboard_payload = json.loads(DASHBOARD_EXPORT.read_text(encoding='utf-8'))
view_model = build_dashboard_view_model(dashboard_payload)
readiness_overview = view_model['readiness_overview']
html = DASHBOARD_HTML.read_text(encoding='utf-8')
summary = {
    'schema_version': 'schedule.dashboard_page_refresh.v1',
    'status': result.status,
    'job_name': config.name,
    'attempts': result.payload.get('attempts'),
    'notifications': events,
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(DASHBOARD_EXPORT),
    'dashboard_page': str(DASHBOARD_HTML),
    'dashboard_schema_version': result.payload.get('dashboard_schema_version'),
    'dashboard_page_schema_version': result.payload.get('dashboard_page_schema_version'),
    'dashboard_view_model_schema_version': result.payload.get('dashboard_view_model_schema_version'),
    'dashboard_page_model_card_count': result.payload.get('dashboard_page_model_card_count'),
    'production_health': str(PRODUCTION_HEALTH),
    'production_health_schema': dashboard_payload['production_health']['schema_version'],
    'production_health_status': dashboard_payload['production_health']['health_status'],
    'notebook_card_manifest': str(NOTEBOOK_CARD_MANIFEST),
    'notebook_card_readiness_schema': dashboard_payload['notebook_card_readiness']['schema_version'],
    'production_published_count': dashboard_payload['notebook_card_readiness']['summary']['production_published_count'],
    'production_template_count': dashboard_payload['notebook_card_readiness']['summary']['production_template_count'],
    'readiness_overview_schema': readiness_overview['schema_version'],
    'readiness_overview_status': readiness_overview['status'],
    'readiness_overview_pass_count': readiness_overview['pass_count'],
    'readiness_overview_review_count': readiness_overview['review_count'],
    'readiness_overview_block_count': readiness_overview['block_count'],
    'schedule_readiness_overview_schema': result.payload.get('readiness_overview_schema_version'),
    'schedule_readiness_overview_status': result.payload.get('readiness_overview_status'),
    'schedule_readiness_overview_pass_count': result.payload.get('readiness_overview_pass_count'),
    'schedule_readiness_overview_review_count': result.payload.get('readiness_overview_review_count'),
    'schedule_readiness_overview_block_count': result.payload.get('readiness_overview_block_count'),
    'schedule_readiness_overview_gates': result.payload.get('readiness_overview_gates'),
    'html_size_bytes': DASHBOARD_HTML.stat().st_size,
    'top_models': [row['model'] for row in dashboard_payload['top_models']],
    'attribution_status': dashboard_payload['attribution']['status'],
    'has_model_filter': 'id="model-filter"' in html,
    'has_regime_filter': 'id="regime-filter"' in html,
    'has_fidelity_filter': 'id="fidelity-filter"' in html,
    'has_view_model_json': 'id="dashboard-view-model"' in html,
    'has_window_regime_table': 'window-regime' in html,
    'has_readiness_overview_panel': 'Readiness Overview' in html,
    'has_notebook_readiness_panel': 'Notebook Readiness' in html,
    'risk_notes': [
        '页面由 schedule 自动刷新生成，仍只消费 dashboard export/view model。',
        'Readiness Overview 汇总 production health、notebook-card readiness、requested model contract 和 model run ledger。',
        'Notebook-card readiness 只说明 notebook/manifest 发布状态，不能替代 production health。',
        '当前是 paper/sim dashboard 页面，不包含真实 broker 下单能力。',
    ],
}
assert summary['status'] == 'success'
assert summary['dashboard_page_schema_version'] == 'dashboard.static_page.v1'
assert summary['dashboard_view_model_schema_version'] == 'dashboard.view_model.v1'
assert summary['dashboard_page_model_card_count'] == len(dashboard_payload['top_models'])
assert summary['has_model_filter']
assert summary['has_regime_filter']
assert summary['has_fidelity_filter']
assert summary['has_view_model_json']
assert summary['has_window_regime_table']
assert summary['has_readiness_overview_panel']
assert summary['has_notebook_readiness_panel']
assert summary['production_health_status'] == 'healthy'
assert summary['readiness_overview_schema'] == 'dashboard.readiness_overview.v1'
assert summary['readiness_overview_status'] == 'review'
assert summary['readiness_overview_pass_count'] >= 2
assert summary['readiness_overview_review_count'] >= 1
assert summary['readiness_overview_block_count'] == 0
assert summary['schedule_readiness_overview_schema'] == summary['readiness_overview_schema']
assert summary['schedule_readiness_overview_status'] == summary['readiness_overview_status']
assert summary['schedule_readiness_overview_pass_count'] == summary['readiness_overview_pass_count']
assert summary['schedule_readiness_overview_review_count'] == summary['readiness_overview_review_count']
assert summary['schedule_readiness_overview_block_count'] == summary['readiness_overview_block_count']
assert summary['schedule_readiness_overview_gates'] == ','.join(row['gate'] for row in readiness_overview['gates'])
assert summary['production_published_count'] == 1
assert summary['production_template_count'] == 1
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'schedule.dashboard_page_refresh.v1',
 'status': 'success',
 'job_name': 'schedule_dashboard_page_refresh_v1',
 'attempts': 1,
 'notifications': ['schedule_dashboard_page_refresh_v1:success'],
 'matrix_card': 'research/schedule_dashboard_page_refresh_v1/matrix_card',
 'dashboard_export': 'research/schedule_dashboard_page_refresh_v1/dashboard_export.json',
 'dashboard_page': 'research/schedule_dashboard_page_refresh_v1/dashboard.html',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'dashboard_page_schema_version': 'dashboard.static_page.v1',
 'dashboard_view_model_schema_version': 'dashboard.view_model.v1',
 'dashboard_page_model_card_count': 5,
 'production_health': 'research/production_registry_health_v1/production_health_export.json',
 'production_health_schema': 'schedule.production_health.v1',
 'production_health_status': 'healthy',
 'notebook_card_manifest': 'research/notebook_card_manifest_v1/notebook_card_manifest.json',
 'notebook_card_readiness_s

## 中文结论

- 做了什么：通过 `src/schedule` 默认流程生成 matrix card、dashboard export 和 `dashboard.html`，并把 production health 与 notebook-card readiness manifest 接入页面。
- 核心指标：job status、attempts、dashboard/page schema、模型卡片数量、HTML size、production health status、dashboard 与 schedule readiness overview、production published/template counts 以 `analysis_summary.json` 为准。
- regime 差异：页面包含 window-regime 表、model_effect_deltas 和 fidelity 过滤，可展示分 regime、分窗口和模型实现层级差异。
- 是否晋升：不晋升 production；这是自动刷新链路验证，不是模型晋升。
- 风险：页面为 paper/sim dashboard；readiness overview 是 dashboard 层汇总，不替代 production health 的路径审计，也不代表真实 broker 下单能力；`gcn_mse` 当前 registry fidelity 为 native_graph，但仍缺真实关系图和更长窗口鲁棒性审计。
- 下一步：把 readiness overview 的 review/block gate 明细做成 dashboard filter 或锚点，让用户能按阻塞项快速定位 production/research 缺口。